# ABTest

[All Quick Tutorials](README.md)

An A/B test randomly assigns units to control and treatment. When the assigned treatment is the only systematic difference, the difference in outcome means estimates its average effect. This example compares user spending after treatment.

## Contents

- [What an A/B test measures](#ab-theory)
- [Compare means](#ab-run)
  - [Inspect group assignments and historical metrics](#ab-inspect-inputs)
  - [Assign roles and configure Welch's test](#ab-roles-welch)
  - [Check group sizes](#ab-group-sizes)
  - [Read the effect estimate and confidence interval](#ab-read-summary)
- [Choose tests for your metric types](#ab-test-types)
  - [Interpret numeric and categorical test results](#ab-read-test-results)
- [Reduce variance with CUPED](#ab-cuped)
  - [Map the outcome to the historical metric](#ab-map-cuped)
  - [Read the variance reduction report](#ab-cuped-report)
- [Adjust for multiple comparisons](#ab-holm)
  - [Define the family of hypotheses](#ab-define-family)
  - [Read the adjusted p-values](#ab-read-adjusted-p)
- [Reduce floating-point precision](#ab-precision)
  - [Check the resulting data type](#ab-check-dtype)
- [Configure CUPAC models](#ab-cupac)
  - [Enable CUPAC and specify model candidates](#ab-select-cupac-models)
  - [Inspect the CUPAC executor](#ab-check-cupac-pipeline)

<a id="ab-theory"></a>

## What an A/B test measures

With independent units and randomized assignment, the difference in means estimates the average effect of assignment. To test $H_0:\mu_T=\mu_C$, Welch's t-test divides the difference by its standard error.

$$
\widehat{\Delta}=\bar Y_T-\bar Y_C,\qquad
t=\frac{\widehat{\Delta}}{\sqrt{s_T^2/n_T+s_C^2/n_C}}.
$$

Here $n_g$ is the sample size and $s_g^2$ the sample variance of group $g$. Welch's test allows the group variances to differ. The effect estimate stays in the original units of the metric. [NIST two-sample t-test](https://www.itl.nist.gov/div898/handbook/eda/section3/eda353.htm)

<a id="ab-run"></a>

## Compare means

<a id="ab-inspect-inputs"></a>

### Inspect group assignments and historical metrics

Inspect the group assignments and the metric values in both periods.

In [1]:
from hypex import ABTest
from hypex.dataset import (
    Dataset,
    ExperimentData,
    PreTargetRole,
    TargetRole,
    TreatmentRole,
)
from hypex.ml import CUPACExecutor
from hypex.utils.tutorial_data_creation import gen_control_variates_df

frame = gen_control_variates_df(
    data_size=1200, dependent_division=False, random_state=42
)
frame[["Treatment", "Target", "Target_lag_1"]].head(3)

,Treatment,Target,Target_lag_1
0,0,1093.358960,389.946418
1,1,725.459061,247.270884
2,0,853.178392,1120.783778


In `Treatment`, 0 denotes control and 1 denotes treatment. `Target_lag_1` is measured before `Target`.

<a id="ab-roles-welch"></a>

### Assign roles and configure Welch's test

Assign `TargetRole` to the post-treatment metric. Give its historical counterpart `PreTargetRole` for the later CUPED example.

`additional_tests` accepts a test name, an `ABTestTypesEnum` member, or a list. The available names are `"t-test"`, `"u-test"`, `"ks-test"`, and `"chi2-test"`. `None` selects the default t-test. An empty list leaves the group comparison without additional tests. For the t-test, `equal_variance=True` assumes equal variances, `False` selects Welch's test, and `None` uses the test's default. Older examples may use `t_test_equal_var` for the same setting.

`ABTest()` is enough for the basic comparison with a group column and outcome. The example spells out `additional_tests="t-test"` and `equal_variance=False` to show the default Welch's t-test.

In [2]:
data = Dataset(
    roles={
        "Treatment": TreatmentRole(),
        "Target": TargetRole(),
        "Target_lag_1": PreTargetRole(parent="Target", lag=1),
    },
    data=frame,
)
result = ABTest(additional_tests="t-test", equal_variance=False).execute(data)
result.sizes

,control size,test size,control size %,test size %,group
1,629,571,52,47,1


<a id="ab-group-sizes"></a>

### Check group sizes

`sizes` shows 629 observations in control and 571 in treatment. Group sizes can differ under random assignment.

<a id="ab-read-summary"></a>

### Read the effect estimate and confidence interval

Open `summary` to read the effect estimate, confidence interval, and p-value.

In [3]:
result.summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value
0,Target,1,452.99182,467.673873,14.682053,3.24113,-12.548254,41.912361,NOT OK,0.290822


The difference in means is about 14.7. The interval from −12.5 to 41.9 includes zero, and the p-value is about 0.29. The data do not establish the direction of the effect.

<a id="ab-test-types"></a>

## Choose tests for your metric types

Pass a list to `additional_tests`. For numeric outcomes, Welch's test compares means, Mann–Whitney compares ranks, and KS compares distributions. For categorical outcomes, the chi-square test compares category proportions. Convert the sample metric into a binary event using a threshold of 500, then examine both metric types in one report.

An explicit `additional_tests` list defines the full set of tests, so include `"t-test"` if you want it. For your own analysis, pass the test or subset you need.

In the full report, look at `feature` and the p-value columns for the relevant tests.

In [4]:
metrics = Dataset(
    roles={
        "Treatment": TreatmentRole(),
        "Target": TargetRole(),
        "converted": TargetRole(str),
    },
    data=frame.assign(
        converted=(frame["Target"] >= 500).map({True: "yes", False: "no"})
    ),
)
tested = ABTest(additional_tests=["t-test", "u-test", "ks-test", "chi2-test"]).execute(
    metrics
)
tested.summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value,KSTest pass,KSTest p-value,UTest pass,UTest p-value,Chi2Test pass,Chi2Test p-value
0,Target,1,452.99182,467.673873,14.682053,3.24113,-12.548254,41.912361,NOT OK,0.290822,NOT OK,0.308668,NOT OK,0.240642,NaN,NaN
1,converted,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NOT OK,0.26478


<a id="ab-read-test-results"></a>

### Interpret numeric and categorical test results

`Target` has results from the numeric tests, and `converted` has the chi-square result. `NaN` indicates that the test was not applied to that metric type. Choose the primary test before reading the report. This example uses several tests to illustrate their interfaces and hypotheses.

<a id="ab-cuped"></a>

## Reduce variance with CUPED

Let $Y$ be the outcome and $X$ a historical covariate with nonzero variance. CUPED subtracts the linear component of $Y$ explained by $X$.

$$
\theta=\frac{\operatorname{Cov}(Y,X)}{\operatorname{Var}(X)},\qquad
Y_{\mathrm{adj}}=Y-\theta(X-\mathbb E[X]).
$$

Centering preserves the population mean. A stronger linear relationship between $Y$ and $X$ allows greater variance reduction. [Deng et al. on CUPED](https://ai.stanford.edu/~ronnyk/2013-02CUPEDImprovingSensitivityOfControlledExperiments.pdf)

<a id="ab-map-cuped"></a>

### Map the outcome to the historical metric

Set `cuped_features` to map `Target` to `Target_lag_1`. The default `None` leaves CUPED disabled.

The `cuped_features` mapping names the covariate directly, so this call can omit `parent` and `lag` on the historical role.

In [5]:
adjusted = ABTest(cuped_features={"Target": "Target_lag_1"}).execute(data)
adjusted.variance_reduction_report

,feature,variance_reduction_pct
0,Target_cuped,10.22413


<a id="ab-cuped-report"></a>

### Read the variance reduction report

The `Target_cuped` row shows about 10% variance reduction for the metric.

<a id="ab-holm"></a>

## Adjust for multiple comparisons

The Holm procedure compares the ordered p-values $p_{(1)}\le\cdots\le p_{(m)}$ in sequence with thresholds $\alpha/(m-i+1)$. It stops at the first $p_{(i)}$ above its threshold. This controls the probability of at least one false rejection in the family of hypotheses. [Holm, 1979](https://www.ime.usp.br/~abe/lista/pdf4R8xPVzCnX.pdf)

<a id="ab-define-family"></a>

### Define the family of hypotheses

If the decision depends on multiple comparisons, define the family of hypotheses in advance. `multitest_method` accepts a string or an `ABNTestMethodsEnum` member. The default is `"holm"`, and `None` also selects it. You can use `"bonferroni"` to control the probability of any false rejection or `"fdr_bh"` to control the false discovery rate under suitable assumptions.

Add a second metric and set `multitest_method="holm"` to make the default choice explicit.

In [6]:
two_metrics = Dataset(
    roles={"Treatment": TreatmentRole(), "Target": TargetRole(), "X": TargetRole()},
    data=frame,
)
corrected = ABTest(multitest_method="holm").execute(two_metrics)
corrected.multitest

,field,test,old p-value,new p-value,correction,rejected,group
0,Target,TTest,0.29082202877767005,0.5816440575553401,0.5,False,1
1,X,TTest,0.736171027548798,0.736171027548798,1.0,False,1


<a id="ab-read-adjusted-p"></a>

### Read the adjusted p-values

For `Target`, the adjustment moves the p-value from about 0.29 to 0.58. At a significance level of 0.05, `rejected` is `False` in both rows.

<a id="ab-precision"></a>

## Reduce floating-point precision

`float32=True` reduces memory by lowering the precision of floating-point columns. The option is off by default. Use `ABTest(float32=True).execute(data)` in a regular run. Here, call the pipeline through `experiment` to see the column type after conversion.

Access the pandas DataFrame through `data` to inspect the physical `dtype`.

In [7]:
before = str(data.select("Target").data.dtypes.iloc[0])
compact = ABTest(float32=True).experiment.execute(ExperimentData(data))
after = str(compact.ds.select("Target").data.dtypes.iloc[0])

{"before": before, "in the pipeline": after}

{'before': 'float64', 'in the pipeline': 'float32'}

<a id="ab-check-dtype"></a>

### Check the resulting data type

The pipeline converts `Target` from `float64` to `float32`, reducing each value from eight bytes to four. For metrics sensitive to rounding, compare estimates and intervals at both precisions.

<a id="ab-cupac"></a>

## Configure CUPAC models

CUPAC uses a prediction $\widehat m(X_{\mathrm{pre}})$ from pretreatment features in place of $X$ in the CUPED formula. The features and training data must be available before treatment.

<a id="ab-select-cupac-models"></a>

### Enable CUPAC and specify model candidates

`enable_cupac=True` enables CUPAC using the time relationships and covariates specified through roles. Optional `cupac_models` limits the model search to a name or a list such as `["linear", "ridge"]`. With the default `None`, HypEx considers the available models.

Inspect the pipeline to see which settings were passed. The [CUPED and CUPAC tutorial](CUPED_CUPAC.ipynb#cupac) works through a full example with two historical periods, roles, and reports.

In [8]:
cupac_test = ABTest(enable_cupac=True, cupac_models=["linear", "ridge"])
next(
    stage.cupac_models
    for stage in cupac_test.experiment.executors
    if isinstance(stage, CUPACExecutor)
)

['linear', 'ridge']

<a id="ab-check-cupac-pipeline"></a>

### Inspect the CUPAC executor

The pipeline contains `CUPACExecutor` with the two candidates `linear` and `ridge`. The output confirms the settings passed to the executor.

See the standalone guides for [multiple comparisons](../how_to_guide/multiple-testing.ipynb) and [CUPED](../how_to_guide/cuped.ipynb).

See how to [run the same A/B test on pandas and Spark](SparkBackend.ipynb#spark-ab).

For several product variants, see the [shared-control A/B/n guide](../how_to_guide/abn-variants.ipynb).

For A/B/n with several metrics, see the [Holm and FDR comparison](../how_to_guide/abn-fdr.ipynb).